# 04. Loss Functions and Optimizers

## 학습 목표

1. Loss Function이 예측과 정답의 차이를 수치화하는 방식을 이해한다.
2. 회귀와 분류에서 사용하는 대표 Loss를 구분한다.
3. Optimizer가 Parameter를 업데이트하는 역할을 이해한다.
4. `zero_grad()`, `backward()`, `step()`의 순서를 익힌다.
5. SGD와 Adam의 기본 차이를 이해한다.
6. 간단한 회귀 및 분류 학습 루프를 구현한다.


In [37]:
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


---

# 1. Loss Function이란?

Loss Function은 모델의 예측값과 실제 정답이 얼마나 다른지를 하나의 숫자로 나타낸다.

```text
예측이 정답에 가까움 → Loss가 작음
예측이 정답에서 멂 → Loss가 큼
```

학습의 목표는 Loss가 작아지는 방향으로 모델 Parameter를 업데이트하는 것이다.


---

# 2. 회귀와 분류

## 회귀

연속적인 숫자를 예측한다.

- 집값
- 온도
- 매출
- 센서값

대표 Loss:

- Mean Squared Error
- Mean Absolute Error

## 분류

정해진 범주를 예측한다.

- 고양이 / 개
- 숫자 0~9
- 정상 / 비정상

대표 Loss:

- Binary Cross Entropy
- Cross Entropy


---

# 3. Mean Squared Error

$$
\operatorname{MSE}
=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat y_i-y_i)^2
$$

- 코드에서 `criterion`이 무엇인가?
  - criterion : (판단이나 결정을 위한) **기준**
  - loss function 객체를 담는 변수명이다. (관습)
  - 왜 그렇게 하냐? loss function 객체의 인수에 설정값을 넣기 때문이다.
    - ex: `criterion = nn.MSELoss(reduction="sum")` 또는 `criterion = nn.MSELoss(reduction="mean")`

In [38]:
prediction = torch.tensor([2.0, 4.0, 6.0])
target = torch.tensor([3.0, 5.0, 5.0])

criterion = nn.MSELoss()
loss = criterion(prediction, target)

print("MSE Loss:", loss)

MSE Loss: tensor(1.)


직접 계산하면:

$$
\frac{(2-3)^2+(4-5)^2+(6-5)^2}{3}=1
$$


In [39]:
manual_loss = ((prediction - target) ** 2).mean()

print("직접 계산:", manual_loss)
print("같은 결과인가?", torch.allclose(loss, manual_loss))

직접 계산: tensor(1.)
같은 결과인가? True


MSE는 오차를 제곱하므로 큰 오차에 더 큰 벌점을 준다.

---

# 4. Mean Absolute Error

$$
\operatorname{MAE}
=
\frac{1}{N}
\sum_{i=1}^{N}
|\hat y_i-y_i|
$$


In [40]:
criterion = nn.L1Loss()
loss = criterion(prediction, target)

print("MAE Loss:", loss)

MAE Loss: tensor(1.)


## MSE와 MAE

- MSE: 큰 오차에 더 민감
- MAE: 이상치의 영향을 상대적으로 덜 받음


---

# 5. Binary Cross Entropy

이진 분류에서는 정답이 보통 0 또는 1이다.


In [41]:
probability = torch.tensor([0.9, 0.2, 0.7])
target = torch.tensor([1.0, 0.0, 1.0])

criterion = nn.BCELoss()
loss = criterion(probability, target)

print("BCE Loss:", loss)

BCE Loss: tensor(0.2284)


실전에서는 Sigmoid와 BCE를 따로 사용하기보다 `BCEWithLogitsLoss`를 주로 사용한다.

이 함수는 내부에서 Sigmoid와 BCE를 수치적으로 안정적으로 결합한다.


In [42]:
logits = torch.tensor([2.2, -1.4, 0.8])
target = torch.tensor([1.0, 0.0, 1.0])

criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, target)

print("Loss:", loss)
print("Sigmoid 확률:", torch.sigmoid(logits))

Loss: tensor(0.2322)
Sigmoid 확률: tensor([0.9002, 0.1978, 0.6900])


`logit`은 Sigmoid 적용 전의 제한 없는 실수값이다.

```text
logit → sigmoid → 0과 1 사이의 확률
```


---

# 6. Multi-class Cross Entropy

여러 클래스 중 하나를 선택하는 분류에서는 `CrossEntropyLoss`를 사용한다.


In [43]:
logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 0.3, 2.2]])

target = torch.tensor([0, 2])

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)

print("CrossEntropy Loss:", loss)

CrossEntropy Loss: tensor(0.3520)


shape 규칙:

```text
logits: (batch_size, num_classes)
target: (batch_size,)
```

Target은 One-hot Vector가 아니라 클래스 번호를 저장한다.


## Softmax를 먼저 적용하지 않는다

`CrossEntropyLoss`는 내부적으로 LogSoftmax와 NLLLoss를 결합한다.

```python
loss = criterion(logits, target)
```

Raw Logit을 직접 전달한다.


In [44]:
probabilities = torch.softmax(logits, dim=1)

print(probabilities)
print("각 행의 합:", probabilities.sum(dim=1))

tensor([[0.6590, 0.2424, 0.0986],
        [0.1371, 0.1123, 0.7506]])
각 행의 합: tensor([1.0000, 1.0000])


---

# 7. `reduction`

여러 샘플의 Loss를 합치는 방식:

- `"mean"`: 평균
- `"sum"`: 합
- `"none"`: 각 Loss를 그대로 반환


In [45]:
prediction = torch.tensor([2.0, 4.0, 6.0])
target = torch.tensor([3.0, 5.0, 5.0])

for reduction in ["none", "mean", "sum"]:
    criterion = nn.MSELoss(reduction=reduction)
    print(reduction, criterion(prediction, target))

none tensor([1., 1., 1.])
mean tensor(1.)
sum tensor(3.)


---

# 8. Optimizer란?

Optimizer는 계산된 Gradient를 사용하여 모델 Parameter를 업데이트한다.

$$
\theta
\leftarrow
\theta-\eta\nabla_{\theta}L
$$

- $\theta$: Parameter
- $\eta$: Learning Rate
- $\nabla_{\theta}L$: Loss의 Gradient


---

# 9. SGD

SGD는 Gradient의 반대 방향으로 Parameter를 이동시킨다.


In [60]:
model = nn.Linear(1, 1)

optimizer = optim.SGD(model.parameters(), lr=0.01)

print(optimizer)

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


Learning Rate가 너무 크면 학습이 불안정하거나 발산할 수 있다. 너무 작으면 학습이 지나치게 느려진다.

---

# 10. Adam

Adam은 Gradient의 이동 평균 등을 사용하여 Parameter별 업데이트 크기를 적응적으로 조절한다.


In [47]:
model = nn.Linear(1, 1)

optimizer = optim.Adam(model.parameters(), lr=0.001)

print(optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


기본적인 경향:

- Adam: 초반 학습이 빠르고 기본값이 비교적 안정적
- SGD: 튜닝이 필요하지만 일부 문제에서 일반화 성능이 좋을 수 있음

항상 한 Optimizer가 더 우수한 것은 아니다.


---

# 11. 기본 학습 루프

```python
optimizer.zero_grad()
prediction = model(x)
loss = criterion(prediction, target)
loss.backward()
optimizer.step()
```

각 단계:

1. 이전 Gradient 초기화
2. Forward
3. Loss 계산
4. Backward
5. Parameter 업데이트


---

# 12. 선형 회귀 학습

다음 관계를 학습한다.

$$
y=3x+2
$$


In [48]:
torch.manual_seed(42)

x = torch.linspace(-2, 2, steps=100).unsqueeze(1)
noise = torch.randn_like(x) * 0.2
y = 3 * x + 2 + noise

print("x shape:", x.shape)
print("y shape:", y.shape)

x shape: torch.Size([100, 1])
y shape: torch.Size([100, 1])


In [49]:
model = nn.Linear(1, 1)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)

for epoch in range(200):
    optimizer.zero_grad()

    prediction = model(x)
    loss = criterion(prediction, y)

    loss.backward()
    optimizer.step()

    if epoch % 40 == 0:
        print(f"epoch={epoch:3d}, " f"loss={loss.item():.6f}")

print("학습된 weight:", model.weight.item())
print("학습된 bias:", model.bias.item())

epoch=  0, loss=19.378107
epoch= 40, loss=0.040435
epoch= 80, loss=0.038517
epoch=120, loss=0.038516
epoch=160, loss=0.038516
학습된 weight: 2.99705171585083
학습된 bias: 2.0119519233703613


학습 결과는 대체로 다음에 가까워진다.

```text
weight ≈ 3
bias ≈ 2
```

데이터에 Noise가 있으므로 정확히 일치하지 않을 수 있다.


---

# 13. Gradient와 Parameter 업데이트 확인


In [50]:
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

x_sample = torch.tensor([[1.0]])
y_sample = torch.tensor([[5.0]])

print("업데이트 전 weight:", model.weight.item())
print("업데이트 전 bias:", model.bias.item())

optimizer.zero_grad()

prediction = model(x_sample)
loss = criterion(prediction, y_sample)
loss.backward()

print("\nweight gradient:", model.weight.grad.item())
print("bias gradient:", model.bias.grad.item())

optimizer.step()

print("\n업데이트 후 weight:", model.weight.item())
print("업데이트 후 bias:", model.bias.item())

업데이트 전 weight: -0.4308730363845825
업데이트 전 bias: -0.5986685752868652

weight gradient: -12.059082984924316
bias gradient: -12.059082984924316

업데이트 후 weight: 0.7750352621078491
업데이트 후 bias: 0.6072397232055664


`backward()`는 Gradient만 계산한다.

실제로 Parameter를 변경하는 것은 `optimizer.step()`이다.


---

# 14. 다중 클래스 분류 예제

2차원 입력을 3개 클래스로 분류한다.


In [51]:
torch.manual_seed(42)

class_0 = torch.randn(100, 2) + torch.tensor([-2.0, 0.0])
class_1 = torch.randn(100, 2) + torch.tensor([2.0, 0.0])
class_2 = torch.randn(100, 2) + torch.tensor([0.0, 2.5])

x = torch.cat([class_0, class_1, class_2], dim=0)
y = torch.cat(
    [
        torch.zeros(100, dtype=torch.long),
        torch.ones(100, dtype=torch.long),
        torch.full((100,), 2, dtype=torch.long),
    ]
)

print("x shape:", x.shape)
print("y shape:", y.shape)
print("클래스:", torch.unique(y))

x shape: torch.Size([300, 2])
y shape: torch.Size([300])
클래스: tensor([0, 1, 2])


In [52]:
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 3))

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    optimizer.zero_grad()

    logits = model(x)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    if epoch % 40 == 0:
        predictions = logits.argmax(dim=1)
        accuracy = (predictions == y).float().mean()

        print(
            f"epoch={epoch:3d}, "
            f"loss={loss.item():.4f}, "
            f"accuracy={accuracy.item():.4f}"
        )

epoch=  0, loss=1.1843, accuracy=0.3367
epoch= 40, loss=0.1919, accuracy=0.9133
epoch= 80, loss=0.1813, accuracy=0.9200
epoch=120, loss=0.1784, accuracy=0.9233
epoch=160, loss=0.1762, accuracy=0.9267


## 예측 클래스

가장 큰 Logit의 인덱스를 클래스 번호로 선택한다.


In [53]:
model.eval()

with torch.no_grad():
    logits = model(x)
    predictions = logits.argmax(dim=1)
    accuracy = (predictions == y).float().mean()

print("최종 정확도:", accuracy.item())

최종 정확도: 0.9266666769981384


---

# 15. 학습 모드와 평가 모드

학습:

```python
model.train()

optimizer.zero_grad()
prediction = model(x)
loss = criterion(prediction, target)
loss.backward()
optimizer.step()
```

평가:

```python
model.eval()

with torch.no_grad():
    prediction = model(x)
```


---

# 16. 자주 발생하는 실수

1. `zero_grad()`를 빼먹어 Gradient가 누적됨
2. `optimizer.step()`을 빼먹어 Parameter가 갱신되지 않음
3. `CrossEntropyLoss` 전에 Softmax를 적용함
4. 다중 클래스 Target을 Float로 만듦
5. 모델과 입력의 Device가 다름
6. 예측값과 Target의 shape이 맞지 않음


---

# 17. 연습문제

## 문제 1

다음 값의 MSE를 계산하자.


In [62]:
prediction = torch.tensor([1.0, 3.0, 5.0])
target = torch.tensor([2.0, 2.0, 8.0])

criterion = nn.MSELoss()
loss = criterion(prediction, target)

print(loss)

tensor(3.6667)


## 문제 2

다음 모델을 SGD로 한 번 업데이트하자.

순서:

1. Gradient 초기화
2. Forward
3. Loss 계산
4. Backward
5. Step


In [63]:
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

x = torch.tensor([[2.0]])
target = torch.tensor([[7.0]])

optimizer.zero_grad()
prediction = model(x)
loss = criterion(prediction, target)
loss.backward()
optimizer.step()

print("prediction:", model(x))
print("weight:", model.weight)
print("bias:", model.bias)

prediction: tensor([[7.0000]], grad_fn=<AddmmBackward0>)
weight: Parameter containing:
tensor([[2.7767]], requires_grad=True)
bias: Parameter containing:
tensor([1.4466], requires_grad=True)


## 문제 3

3개 클래스 분류를 위한 Logit과 Target을 만들자.

- Batch Size: 4
- Class 수: 3


In [64]:
logits = torch.randn(4, 3)
target = torch.tensor([0, 2, 1, 0], dtype=torch.long)

print("logits shape:", logits.shape if logits is not None else None)
print("target shape:", target.shape if target is not None else None)


criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)

print("loss:", loss)

logits shape: torch.Size([4, 3])
target shape: torch.Size([4])
loss: tensor(1.8115)


---

# 18. 연습문제 정답


In [57]:
# 문제 1
prediction = torch.tensor([1.0, 3.0, 5.0])
target = torch.tensor([2.0, 2.0, 8.0])

criterion = nn.MSELoss()
loss = criterion(prediction, target)

print(loss)

tensor(3.6667)


$$
\frac{(1-2)^2+(3-2)^2+(5-8)^2}{3}
=
\frac{11}{3}
$$


In [58]:
# 문제 2
model = nn.Linear(1, 1)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

x = torch.tensor([[2.0]])
target = torch.tensor([[7.0]])

optimizer.zero_grad()

prediction = model(x)
loss = criterion(prediction, target)

loss.backward()
optimizer.step()

print("prediction after update:", model(x))
print("weight:", model.weight)
print("bias:", model.bias)

prediction after update: tensor([[7.]], grad_fn=<AddmmBackward0>)
weight: Parameter containing:
tensor([[2.8198]], requires_grad=True)
bias: Parameter containing:
tensor([1.3604], requires_grad=True)


In [59]:
# 문제 3
logits = torch.randn(4, 3)
target = torch.tensor([0, 2, 1, 0], dtype=torch.long)

print("logits shape:", logits.shape)
print("target shape:", target.shape)

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, target)

print("loss:", loss)

logits shape: torch.Size([4, 3])
target shape: torch.Size([4])
loss: tensor(1.2035)


---

# 19. 핵심 정리

## Loss

- 회귀: `MSELoss`, `L1Loss`
- 이진 분류: `BCEWithLogitsLoss`
- 다중 클래스 분류: `CrossEntropyLoss`

## Optimizer

- SGD: 기본적인 Gradient 기반 업데이트
- Adam: 적응적인 업데이트 크기 사용

## 학습 순서

```python
optimizer.zero_grad()
prediction = model(x)
loss = criterion(prediction, target)
loss.backward()
optimizer.step()
```

## 평가 순서

```python
model.eval()

with torch.no_grad():
    prediction = model(x)
```
